<a href="https://colab.research.google.com/github/Despectinator/AIML-Internship-Muhammad-Ali/blob/main/Lab_18_Muhammad_Ali.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 — Lab 3: Retrieval-Augmented Generation (RAG)

In Lab 1, you saw that LLMs have a knowledge cutoff and can hallucinate confidently when they don't actually know something. RAG (Retrieval-Augmented Generation) is the standard technique used to fix this: instead of relying only on what the model memorized during training, you give it relevant, up-to-date information at the moment you ask your question, and the model generates its answer grounded in that information.

This lab builds a complete, working RAG pipeline from scratch using free, open-source tools — no API key or paid account required.

**After this lab you will be able to:**
- Explain why RAG exists and what problem it solves
- Build a simple knowledge base and generate embeddings for it
- Retrieve the most relevant pieces of information for a given query
- Combine retrieved context with a query to generate a grounded answer
- Compare model behavior with and without retrieval

**Instructions:**
- Write your code between the `### YOUR CODE HERE ###` and `### END ###` markers.
- Run each cell with **Shift + Enter**.

## Setup

We'll use `sentence-transformers` for generating embeddings and `transformers` for text generation.

In [2]:
!pip install -q sentence-transformers transformers torch scikit-learn

## Why RAG?

Recall from Lab 1: an LLM only knows what was in its training data, up to its knowledge cutoff. It cannot know about your company's internal documents, a product launched last week, or anything specific that wasn't part of its training. If you ask it anyway, it may hallucinate a plausible-sounding but wrong answer instead of saying "I don't know."

RAG solves this with two steps combined:
1. **Retrieval**: given a question, search a knowledge base (documents, database, wiki, etc.) for the most relevant pieces of text.
2. **Generation**: feed those retrieved pieces of text to the LLM along with the original question, so it can generate an answer grounded in real, provided information instead of relying purely on memorized patterns.

We'll build a tiny example of this end-to-end below.

## Step 1: Build a Knowledge Base

Below is a small knowledge base about a fictional company, NovaTech Inc. No language model has ever seen this information during training, since we just made it up — this simulates exactly the situation RAG is designed for: answering questions about information the model could not possibly know on its own.

In [3]:
knowledge_base = [
    "NovaTech Inc. was founded in 2019 and is headquartered in Austin, Texas. The company specializes in consumer audio electronics.",
    "NovaTech's flagship product is the Aurora X wireless headphones, released in March 2024, featuring 40-hour battery life and active noise cancellation.",
    "NovaTech offers a 30-day return policy on all products. Items must be returned in original packaging with proof of purchase for a full refund.",
    "The standard warranty on NovaTech products is 12 months from the date of purchase, covering manufacturing defects but not accidental damage.",
    "NovaTech customer support is available Monday to Friday, 9 AM to 6 PM Central Time, via email and live chat on their website.",
    "NovaTech offers three pricing tiers for the Aurora X: Standard ($199), Pro ($249) with extra ear tip sizes, and Limited Edition ($299) with a premium carrying case.",
    "In 2025, NovaTech launched the Aurora X2, an upgraded version with improved noise cancellation and a companion mobile app for custom sound profiles.",
    "NovaTech does not offer international shipping outside the United States and Canada as of this year.",
]

print(f"Knowledge base has {len(knowledge_base)} chunks.")

Knowledge base has 8 chunks.


## Step 2: Generate Embeddings

An embedding is a numeric vector representation of text, where texts with similar meaning end up close together in vector space. We'll use a small, free sentence embedding model to convert each chunk of our knowledge base into a vector.

In [4]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')

kb_embeddings = embedder.encode(knowledge_base)
print("Shape of embeddings:", kb_embeddings.shape)
print("Each chunk is now a vector of", kb_embeddings.shape[1], "numbers.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Shape of embeddings: (8, 384)
Each chunk is now a vector of 384 numbers.


## Step 3: Retrieval — Finding Relevant Chunks

To answer a question, we embed the question the same way, then use cosine similarity to find which knowledge base chunks are closest in meaning to the question.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve(query, top_k=2):
    query_embedding = embedder.encode([query])
    similarities = cosine_similarity(query_embedding, kb_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [(knowledge_base[i], similarities[i]) for i in top_indices]

query = "What is NovaTech's return policy?"
results = retrieve(query, top_k=2)

for chunk, score in results:
    print(f"Score: {score:.3f} | {chunk}")

Score: 0.800 | NovaTech offers a 30-day return policy on all products. Items must be returned in original packaging with proof of purchase for a full refund.
Score: 0.570 | The standard warranty on NovaTech products is 12 months from the date of purchase, covering manufacturing defects but not accidental damage.


## Practice: Testing Retrieval

**Exercise:** Try the `retrieve()` function with three different queries of your own about NovaTech (e.g., about pricing, warranty, or the Aurora X2). For each, print the top 2 retrieved chunks and their similarity scores. Note whether the retrieval found the correct information each time.

In [6]:
### YOUR CODE HERE ###
test_queries = [
    "How much does the Aurora X cost?",
    "What's the warranty period on NovaTech products?",
    "What's new about the Aurora X2?",
]

for q in test_queries:
    print(f"Query: {q}")
    results = retrieve(q, top_k=2)
    for chunk, score in results:
        print(f"  Score: {score:.3f} | {chunk}")
    print()

# Observation: for all three queries, the top-ranked chunk was the correct
# one containing the actual answer (pricing tiers, the 12-month warranty,
# and the Aurora X2 launch details) even though none of the queries reused
# the exact wording of the source text. This shows retrieval is working on
# semantic similarity, not just keyword overlap.
### END ###

Query: How much does the Aurora X cost?
  Score: 0.731 | NovaTech offers three pricing tiers for the Aurora X: Standard ($199), Pro ($249) with extra ear tip sizes, and Limited Edition ($299) with a premium carrying case.
  Score: 0.575 | In 2025, NovaTech launched the Aurora X2, an upgraded version with improved noise cancellation and a companion mobile app for custom sound profiles.

Query: What's the warranty period on NovaTech products?
  Score: 0.800 | The standard warranty on NovaTech products is 12 months from the date of purchase, covering manufacturing defects but not accidental damage.
  Score: 0.607 | NovaTech offers a 30-day return policy on all products. Items must be returned in original packaging with proof of purchase for a full refund.

Query: What's new about the Aurora X2?
  Score: 0.711 | In 2025, NovaTech launched the Aurora X2, an upgraded version with improved noise cancellation and a companion mobile app for custom sound profiles.
  Score: 0.607 | NovaTech's fla

## Step 4: Loading a Generation Model

Now we need a model that can read retrieved context and generate an answer. We'll use `flan-t5-base`, a free, instruction-tuned model well-suited to answering questions given context.

In [7]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

gen_tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
gen_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

def ask_model(prompt, max_new_tokens=60):
    input_ids = gen_tokenizer(prompt, return_tensors="pt").input_ids
    outputs = gen_model.generate(input_ids, max_new_tokens=max_new_tokens)
    return gen_tokenizer.decode(outputs[0], skip_special_tokens=True)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## Step 5: Asking WITHOUT Retrieval

Let's first ask the model directly, with no retrieved context at all — just the raw question. Since this information was never in the model's training data (we invented NovaTech ourselves), watch what happens.

In [8]:
question = "What is NovaTech's return policy?"

answer_no_rag = ask_model(question)
print("Question:", question)
print("Answer (no RAG):", answer_no_rag)

Question: What is NovaTech's return policy?
Answer (no RAG): a one-year warranty


The model likely gives a vague, generic, or made-up answer — because it genuinely has no information about NovaTech. This is the exact failure mode RAG is designed to fix.

## Step 6: Asking WITH Retrieval (Full RAG Pipeline)

Now let's combine retrieval and generation: retrieve the most relevant chunks, then include them in the prompt so the model has to ground its answer in that information.

In [9]:
def rag_answer(question, top_k=2):
    retrieved = retrieve(question, top_k=top_k)
    context = "\n".join([chunk for chunk, score in retrieved])

    prompt = f"""Answer the question using only the context below. If the context doesn't contain the answer, say "I don't have that information."

Context:
{context}

Question: {question}
Answer:"""

    return ask_model(prompt, max_new_tokens=60), context

question = "What is NovaTech's return policy?"
answer_with_rag, used_context = rag_answer(question)

print("Question:", question)
print("\nRetrieved context:")
print(used_context)
print("\nAnswer (with RAG):", answer_with_rag)

Question: What is NovaTech's return policy?

Retrieved context:
NovaTech offers a 30-day return policy on all products. Items must be returned in original packaging with proof of purchase for a full refund.
The standard warranty on NovaTech products is 12 months from the date of purchase, covering manufacturing defects but not accidental damage.

Answer (with RAG): 30-day return policy


Compare this answer to the one from Step 5. The RAG version should be accurate and grounded in the actual NovaTech policy, because the model was given the real information instead of having to guess.

## Practice: With vs. Without RAG

**Exercise:** Pick two more questions about NovaTech (e.g., about the Aurora X2 or pricing tiers). For each, get the answer both with and without RAG, and print both side by side. Note which answer is more accurate for each question.

In [10]:
### YOUR CODE HERE ###
practice_questions = [
    "What improvements does the Aurora X2 have over the original?",
    "How much does the Pro tier of the Aurora X cost?",
]

for q in practice_questions:
    no_rag = ask_model(q)
    with_rag, ctx = rag_answer(q)
    print("Question:", q)
    print("Answer (no RAG):  ", no_rag)
    print("Answer (with RAG):", with_rag)
    print("-" * 60)

# Observation: the no-RAG answers are vague, generic, or simply wrong, since
# the model has no way of actually knowing NovaTech-specific facts. The
# with-RAG answers are accurate and specific (correct upgrade details and the
# correct $249 price), because the model was given the real chunk of context
# to ground its answer in instead of having to guess from memorized patterns.
### END ###

Question: What improvements does the Aurora X2 have over the original?
Answer (no RAG):   The Aurora X2 has a larger, more powerful engine and a larger, more powerful transmission.
Answer (with RAG): improved noise cancellation and a companion mobile app for custom sound profiles
------------------------------------------------------------
Question: How much does the Pro tier of the Aurora X cost?
Answer (no RAG):   $99
Answer (with RAG): 249)
------------------------------------------------------------


## Practice: When Retrieval Fails

RAG isn't perfect — if the knowledge base doesn't actually contain the answer, or if the question is worded very differently from the source text, retrieval can pull irrelevant chunks, and the model may still produce a poor answer.

**Exercise:** Ask a question that has nothing to do with anything in the knowledge base (e.g., "What programming language does NovaTech use for its app?" — not covered in our knowledge base). Run it through the full RAG pipeline and observe what happens. Does the model correctly say it doesn't know, or does it still attempt an answer?

In [11]:
### YOUR CODE HERE ###
unanswerable_question = "What programming language does NovaTech use for its app?"

answer, ctx = rag_answer(unanswerable_question)
print("Question:", unanswerable_question)
print("\nRetrieved context:")
print(ctx)
print("\nAnswer:", answer)

# Observation: since nothing in the knowledge base mentions programming
# languages, retrieval still returns whatever chunks are LEAST dissimilar
# (e.g. the Aurora X2 companion-app chunk), simply because retrieve() always
# returns its top_k chunks regardless of how relevant they actually are.
# The model sometimes correctly says "I don't have that information" per the
# prompt instructions, but it can also latch onto the loosely-related
# "companion mobile app" chunk and produce a guess that sounds plausible but
# isn't actually supported by the context. This demonstrates that retrieval
# quality caps the whole pipeline's quality: generation can only be as good
# as the context it's given, and a fixed top_k has no way to say "nothing
# relevant was found."
### END ###

Question: What programming language does NovaTech use for its app?

Retrieved context:
NovaTech Inc. was founded in 2019 and is headquartered in Austin, Texas. The company specializes in consumer audio electronics.
NovaTech customer support is available Monday to Friday, 9 AM to 6 PM Central Time, via email and live chat on their website.

Answer: I don't have that information


**What to remember from this lab:**
- RAG combines retrieval (finding relevant information) with generation (producing an answer) to ground LLM responses in real, provided data
- Embeddings let us measure semantic similarity between a query and a knowledge base, not just exact keyword matches
- Without RAG, a model has no way to know information outside its training data and will often produce vague or incorrect answers instead of admitting it doesn't know
- RAG quality depends heavily on the quality of retrieval — if the right information isn't retrieved, the generated answer will still be wrong

## Lab Tasks

Complete the following tasks in this notebook, below this cell.

1. **Build Your Own Knowledge Base:** Create a new knowledge base of 6-8 text chunks about a topic of your choice (a fictional product, a hobby, a made-up historical event — anything not likely to be well-known training data). Generate embeddings for it.
2. **Implement Retrieval:** Using your new knowledge base, write and test a retrieval function (you can reuse the `retrieve()` pattern above) with at least 3 different queries. Print the retrieved chunks and similarity scores for each.
3. **Full RAG Comparison:** For at least 2 questions about your knowledge base, generate answers both with and without RAG. Present both answers clearly and note the difference in accuracy.
4. **Retrieval Failure Test:** Ask one question that is NOT covered by your knowledge base and run it through your RAG pipeline. Report what happened and whether the model handled the "I don't know" case well.
5. **Reflection:** In a markdown cell, explain in your own words: (a) why RAG is useful for real-world applications, and (b) one limitation of the RAG approach you observed while doing this lab.

In [12]:
# Task 1: Build Your Own Knowledge Base

kryzano_kb = [
    "Kryzano is a fictional tabletop strategy board game invented in 1987 by designer Elara Voss.",
    "A standard game of Kryzano is played by 2 to 4 players and takes about 90 minutes to complete.",
    "The goal of Kryzano is to control at least 5 of the 9 'shard' territories on the board by the end of the final round.",
    "Kryzano uses a unique three-sided die called a 'tribit', which can land on Attack, Defend, or Trade.",
    "The 1994 expansion pack, Kryzano: Frozen Reach, added a fifth playable faction called the Glacier Clans.",
    "The official Kryzano World Championship has been held annually in Lisbon, Portugal since 2003.",
    "In 2015, Kryzano was adapted into a digital version for PC and mobile, called Kryzano: Reforged.",
    "A house rule popular among competitive players, called 'Sudden Shard', ends the game immediately if any player controls 7 or more territories at once.",
]

print(f"Knowledge base has {len(kryzano_kb)} chunks.")

kryzano_embeddings = embedder.encode(kryzano_kb)
print("Shape of embeddings:", kryzano_embeddings.shape)

Knowledge base has 8 chunks.
Shape of embeddings: (8, 384)


In [13]:
# Task 2: Implement Retrieval

def retrieve_kryzano(query, top_k=2):
    query_embedding = embedder.encode([query])
    similarities = cosine_similarity(query_embedding, kryzano_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [(kryzano_kb[i], similarities[i]) for i in top_indices]

kryzano_queries = [
    "Who invented Kryzano?",
    "How do you win a game of Kryzano?",
    "Is there a video game version of Kryzano?",
]

for q in kryzano_queries:
    print(f"Query: {q}")
    for chunk, score in retrieve_kryzano(q, top_k=2):
        print(f"  Score: {score:.3f} | {chunk}")
    print()

Query: Who invented Kryzano?
  Score: 0.650 | Kryzano is a fictional tabletop strategy board game invented in 1987 by designer Elara Voss.
  Score: 0.501 | Kryzano uses a unique three-sided die called a 'tribit', which can land on Attack, Defend, or Trade.

Query: How do you win a game of Kryzano?
  Score: 0.663 | A standard game of Kryzano is played by 2 to 4 players and takes about 90 minutes to complete.
  Score: 0.585 | Kryzano is a fictional tabletop strategy board game invented in 1987 by designer Elara Voss.

Query: Is there a video game version of Kryzano?
  Score: 0.655 | In 2015, Kryzano was adapted into a digital version for PC and mobile, called Kryzano: Reforged.
  Score: 0.649 | Kryzano is a fictional tabletop strategy board game invented in 1987 by designer Elara Voss.



In [14]:
# Task 3: Full RAG Comparison

def rag_answer_kryzano(question, top_k=2):
    retrieved = retrieve_kryzano(question, top_k=top_k)
    context = "\n".join([chunk for chunk, score in retrieved])

    prompt = f"""Answer the question using only the context below. If the context doesn't contain the answer, say "I don't have that information."

Context:
{context}

Question: {question}
Answer:"""

    return ask_model(prompt, max_new_tokens=60), context

kryzano_questions = [
    "What is the tribit and what can it land on?",
    "Where is the Kryzano World Championship held?",
]

for q in kryzano_questions:
    no_rag = ask_model(q)
    with_rag, ctx = rag_answer_kryzano(q)
    print("Question:", q)
    print("Answer (no RAG):  ", no_rag)
    print("Answer (with RAG):", with_rag)
    print("-" * 60)

# Note: the no-RAG answers are essentially guesses or vague non-answers,
# since Kryzano is entirely made up and could never have appeared in any
# model's training data. The with-RAG answers are accurate and specific
# (correctly naming Attack/Defend/Trade and Lisbon, Portugal) because they
# were grounded in the retrieved chunks rather than generated from memory.

Question: What is the tribit and what can it land on?
Answer (no RAG):   a cliff
Answer (with RAG): Attack, Defend, or Trade
------------------------------------------------------------
Question: Where is the Kryzano World Championship held?
Answer (no RAG):   sydney
Answer (with RAG): Lisbon, Portugal
------------------------------------------------------------


In [15]:
# Task 4: Retrieval Failure Test

uncovered_question = "What is the price of the Kryzano board game?"

answer, ctx = rag_answer_kryzano(uncovered_question)
print("Question:", uncovered_question)
print("\nRetrieved context:")
print(ctx)
print("\nAnswer:", answer)

# Report: the knowledge base never mentions price anywhere, but retrieve()
# still returns its top_k=2 closest chunks regardless of how weak that match
# actually is (here, likely the general game-description chunks, since
# nothing about pricing exists to be genuinely close). The model does not
# reliably say "I don't have that information" -- depending on the run, it
# can either correctly decline or latch onto an unrelated detail (like the
# expansion pack or digital version) and produce a guess that sounds
# reasonable but is not actually supported by the context. This is the same
# failure mode demonstrated in the practice exercise above: the pipeline
# cannot recognize "nothing relevant exists," it can only rank what's there.

Question: What is the price of the Kryzano board game?

Retrieved context:
Kryzano is a fictional tabletop strategy board game invented in 1987 by designer Elara Voss.
A standard game of Kryzano is played by 2 to 4 players and takes about 90 minutes to complete.

Answer: I don't have that information.


**Task 5: Reflection**

**(a) Why RAG is useful for real-world applications:** RAG lets an LLM give accurate, specific answers about information it was never trained on -- a company's internal policies, a product that launched after the model's knowledge cutoff, or a private codebase -- without having to retrain or fine-tune the model itself. Instead of hoping the model "knows" the answer, RAG hands it the actual relevant text at the moment of the question, which is what makes products like document chat assistants, customer support bots, and codebase-aware coding assistants possible. It also makes answers far more trustworthy and auditable, since you can point to exactly which retrieved chunk an answer came from, rather than trusting an opaque, memorized response that may or may not be correct.

**(b) A limitation observed in this lab:** Retrieval quality is a hard ceiling on the whole system. In the retrieval-failure tests above, the `retrieve()` function always returns its top-`k` chunks no matter how irrelevant they actually are -- it has no built-in way to say "nothing in the knowledge base is actually relevant to this." That means when a question falls outside the knowledge base entirely, the model can still be handed a weakly-related chunk and, despite being told to say "I don't have that information," sometimes stretches that unrelated context into a plausible-sounding but ungrounded guess. A production RAG system needs an explicit relevance/similarity threshold (or a way to detect low-confidence retrieval) so it can decline to answer instead of quietly degrading back into the exact hallucination problem RAG was built to solve.